# SQL Agent for Spider text-to-SQL benchmark

This notebook demonstrates a basic SQL agent that translates natural language questions into SQL queries.

## Environment

For this demo, we use a SQLite database environment based on a standard text-to-sql benchmark called [Spider](https://yale-lily.github.io/spider). The environment provides a gym-like interface and can be used as follows.

In [14]:
%pip install spider-env


Note: you may need to restart the kernel to use updated packages.


In [15]:
!pip uninstall autogen pyautogen -y
!pip install pyautogen==0.2.23

Found existing installation: pyautogen 0.2.23
Uninstalling pyautogen-0.2.23:
  Successfully uninstalled pyautogen-0.2.23


  Using cached pyautogen-0.2.23-py3-none-any.whl.metadata (21 kB)
Using cached pyautogen-0.2.23-py3-none-any.whl (244 kB)


In [16]:
import json
import os
from typing import Annotated, Dict

from spider_env import SpiderEnv

from autogen import ConversableAgent, UserProxyAgent, config_list_from_json

gym = SpiderEnv()

# Randomly select a question from Spider
observation, info = gym.reset()

Loading cached Spider dataset from C:\Users\naman/.cache/spider
Schema file not found for C:\Users\naman/.cache/spider\spider/database/epinions_1
Schema file not found for C:\Users\naman/.cache/spider\spider/database/chinook_1
Schema file not found for C:\Users\naman/.cache/spider\spider/database/company_1
Schema file not found for C:\Users\naman/.cache/spider\spider/database/twitter_1
Schema file not found for C:\Users\naman/.cache/spider\spider/database/icfp_1
Schema file not found for C:\Users\naman/.cache/spider\spider/database/small_bank_1
Schema file not found for C:\Users\naman/.cache/spider\spider/database/flight_4


In [17]:
# The natural language question
question = observation["instruction"]
print(question)

Find the famous titles of artists that do not have any volume.


In [18]:
# The schema of the corresponding database
schema = info["schema"]
print(schema)

CREATE TABLE "artist" (
"Artist_ID" int,
"Artist" text,
"Age" int,
"Famous_Title" text,
"Famous_Release_date" text,
PRIMARY KEY ("Artist_ID")
);
CREATE TABLE "volume" (
"Volume_ID" int,
"Volume_Issue" text,
"Issue_Date" text,
"Weeks_on_Top" real,
"Song" text,
"Artist_ID" int,
PRIMARY KEY ("Volume_ID"),
FOREIGN KEY ("Artist_ID") REFERENCES "artist"("Artist_ID")
);
CREATE TABLE "music_festival" (
"ID" int,
"Music_Festival" text,
"Date_of_ceremony" text,
"Category" text,
"Volume" int,
"Result" text,
PRIMARY KEY ("ID"),
FOREIGN KEY ("Volume") REFERENCES "volume"("Volume_ID")
);



## Agent Implementation

Using AutoGen, a SQL agent can be implemented with a ConversableAgent. The gym environment executes the generated SQL query and the agent can take execution results as feedback to improve its generation in multiple rounds of conversations.

In [19]:
import autogen
print(autogen.__version__)
print(autogen.__file__)

0.2.23
c:\Users\naman\anaconda3\envs\autogen_env\Lib\site-packages\autogen\__init__.py


In [20]:
from autogen import AssistantAgent



In [32]:
os.environ["AUTOGEN_USE_DOCKER"] = "False"
# config_list = config_list_from_json(env_or_file="OAI_CONFIG_LIST")
# config_list = [
#     {
#         "model": "gpt-4o-mini",
#         "api_key": "sk-ijklmnopqrstuvwxijklmnopqrstuvwxijklmnop"
#     }
# ]



def check_termination(msg: Dict):
    if "tool_responses" not in msg:
        return False
    json_str = msg["tool_responses"][0]["content"]
    obj = json.loads(json_str)
    return ("error" not in obj or obj["error"] is None) and obj.get("reward", 0) == 1
from autogen import AssistantAgent, UserProxyAgent

config_list = [
    {
        "model": "qwen3:4b",
        "base_url": "http://localhost:11434/v1",
        "api_key": "ollama"
    }
]

llm_config = {
    "config_list": config_list,
    "temperature": 0,
}
sql_writer = ConversableAgent(
    "sql_writer",
    llm_config=llm_config,
    system_message="You are good at writing SQL queries.",
    is_termination_msg=check_termination,
)

user_proxy = UserProxyAgent(
    "user_proxy",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=5
)

@sql_writer.register_for_llm(description="Function for executing SQL query and returning a response")
@user_proxy.register_for_execution()
def execute_sql(
    reflection: Annotated[str, "Think about what to do"], sql: Annotated[str, "SQL query"]
) -> Annotated[Dict[str, str], "Dictionary with keys 'result' and 'error'"]:
    observation, reward, _, _, info = gym.step(sql)
    error = observation["feedback"]["error"]
    if not error and reward == 0:
        error = "The SQL query returned an incorrect result"
    if error:
        return {
            "error": error,
            "wrong_result": observation["feedback"]["result"],
            "correct_result": info["gold_result"],
        }
    else:
        return {
            "result": observation["feedback"]["result"],
        }

The agent can then take as input the schema and the text question, and generate the SQL query.

In [35]:
message = f"""Below is the schema for a SQL database:
{schema}
Generate a SQL query to answer the following question:
{question}
"""

def safe_check_termination(msg: Dict):
    tool_responses = msg.get("tool_responses", [])
    if not tool_responses:
        return False

    content = tool_responses[0].get("content", "")
    if not content:
        return False

    try:
        obj = json.loads(content) if isinstance(content, str) else content
    except json.JSONDecodeError:
        return False

    return ("error" not in obj or obj["error"] is None) and obj.get("reward", 0) == 1

# Patch termination check to avoid JSONDecodeError on non-JSON/empty tool responses
sql_writer._is_termination_msg = safe_check_termination

chat_result = user_proxy.initiate_chat(sql_writer, message=message)
chat_result

user_proxy (to sql_writer):

Below is the schema for a SQL database:
CREATE TABLE "artist" (
"Artist_ID" int,
"Artist" text,
"Age" int,
"Famous_Title" text,
"Famous_Release_date" text,
PRIMARY KEY ("Artist_ID")
);
CREATE TABLE "volume" (
"Volume_ID" int,
"Volume_Issue" text,
"Issue_Date" text,
"Weeks_on_Top" real,
"Song" text,
"Artist_ID" int,
PRIMARY KEY ("Volume_ID"),
FOREIGN KEY ("Artist_ID") REFERENCES "artist"("Artist_ID")
);
CREATE TABLE "music_festival" (
"ID" int,
"Music_Festival" text,
"Date_of_ceremony" text,
"Category" text,
"Volume" int,
"Result" text,
PRIMARY KEY ("ID"),
FOREIGN KEY ("Volume") REFERENCES "volume"("Volume_ID")
);

Generate a SQL query to answer the following question:
Find the famous titles of artists that do not have any volume.


--------------------------------------------------------------------------------

>>>>>>>> USING AUTO REPLY...
sql_writer (to user_proxy):

SELECT artist.Famous_Title
FROM artist
LEFT JOIN volume ON artist.Artist_ID = volume.Arti

ChatResult(chat_id=None, chat_history=[{'content': 'Below is the schema for a SQL database:\nCREATE TABLE "artist" (\n"Artist_ID" int,\n"Artist" text,\n"Age" int,\n"Famous_Title" text,\n"Famous_Release_date" text,\nPRIMARY KEY ("Artist_ID")\n);\nCREATE TABLE "volume" (\n"Volume_ID" int,\n"Volume_Issue" text,\n"Issue_Date" text,\n"Weeks_on_Top" real,\n"Song" text,\n"Artist_ID" int,\nPRIMARY KEY ("Volume_ID"),\nFOREIGN KEY ("Artist_ID") REFERENCES "artist"("Artist_ID")\n);\nCREATE TABLE "music_festival" (\n"ID" int,\n"Music_Festival" text,\n"Date_of_ceremony" text,\n"Category" text,\n"Volume" int,\n"Result" text,\nPRIMARY KEY ("ID"),\nFOREIGN KEY ("Volume") REFERENCES "volume"("Volume_ID")\n);\n\nGenerate a SQL query to answer the following question:\nFind the famous titles of artists that do not have any volume.\n', 'role': 'assistant'}, {'content': 'SELECT artist.Famous_Title\nFROM artist\nLEFT JOIN volume ON artist.Artist_ID = volume.Artist_ID\nWHERE volume.Artist_ID IS NULL;', 'role'